#### [PREV](9.OPP_Decision_Trees.ipynb) | [HOME](../README.md)

---

# OOP K-Means Clustering Implementation

This Jupyter Notebook provides a basic implementation of K-Means Clustering. The data set contains measurements of 150 iris flowers from three species. We want to find groups of similar flowers using only their petal length and petal width, without telling the model which species each flower belongs to.

#### Supervised vs Unsupervised Learning

Notebooks 1 to 9 use **supervised learning**: the model is trained on features *and* a target, so it learns to predict a known answer. K-Means uses **unsupervised learning**: the model is trained on features only and finds groups (clusters) of similar data points by itself.

| | Supervised (notebooks 1 to 9) | Unsupervised (this notebook) |
|:--|:--|:--|
| Training data | Features and a target | Features only |
| Goal | Predict a value or a class | Find groups in the data |
| Example | KNN predicts an insect's sex | K-Means groups flowers by petal size |
| Code | `model.fit(X, y)` | `model.fit(X)` |

Unsupervised learning is useful when labelled data does not exist or is expensive to produce. Examples include grouping customers by what they buy, grouping news articles by topic and finding unusual network traffic.

#### How K-Means Works

1. Choose $k$, the number of clusters.
2. Choose $k$ starting **centroids**. A centroid is the centre point of a cluster.
3. **Assign:** each data point joins the cluster of its nearest centroid.
4. **Update:** each centroid moves to the mean (average position) of the points in its cluster.
5. Repeat steps 3 and 4 until the centroids stop moving.

```
BEGIN KMeans(data, k)
    centroids = k starting points chosen from data
    REPEAT
        FOR EACH point IN data
            Assign point to the cluster of its nearest centroid
        NEXT point
        FOR EACH cluster
            Move the cluster's centroid to the mean of its points
        NEXT cluster
    UNTIL centroids stop moving
    RETURN centroids
END KMeans
```

Step 10 at the end of this notebook runs these steps one at a time so you can see the centroids move.

#### K-Means vs K-Nearest Neighbour

K-Means and K-Nearest Neighbour (notebook 5) are easy to mix up. Both measure the distance between data points, but they solve different problems.

| | K-Nearest Neighbour | K-Means |
|:--|:--|:--|
| Type | Supervised classification | Unsupervised clustering |
| What $k$ means | Number of neighbours that vote | Number of clusters to find |
| Needs a target | Yes | No |
| What `fit` does | Stores the training data | Moves the centroids until they stop moving |

#### Step 1: Import Dependencies

In [ ]:
import numpy as np
import pandas as pd

%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import silhouette_score

#### Step 2: Import And Visualise The Data

In [ ]:
# Import data as a Pandas DataFrame
iris = pd.read_csv('./data/Iris.csv')

# Features variable (the two petal measurements)
X_iris = iris[['PetalLengthCm', 'PetalWidthCm']]

# There is no target variable in unsupervised learning.
# The species are kept aside and only used in Step 8 to check the clusters.
species = iris['Species']

iris.head()

In [ ]:
# Plot the data the way the model sees it: no species, so every flower is the same colour
plt.scatter(X_iris.PetalLengthCm, X_iris.PetalWidthCm, color='g')
plt.xlabel("Petal length (cm)")
plt.ylabel("Petal width (cm)")
plt.title("Unlabelled Data")
plt.show()

This is all the model gets. Before you continue, decide how many groups you can see. Setosa flowers have small petals and form a clear group in the bottom left. Versicolor and virginica flowers are larger and their groups overlap.

#### Step 3: Scale The Features

Like KNN, K-Means measures the distance between data points. Petal length varies more than petal width (a standard deviation of about 1.76 cm compared with 0.76 cm), so without scaling, petal length would have more influence on the distance. `StandardScaler` rescales each feature to a mean of 0 and a standard deviation of 1, so both features have an equal say.

In [ ]:
scaler = StandardScaler()

# fit_transform learns the mean and standard deviation of each feature, then scales the data
X_scaled = scaler.fit_transform(X_iris)

print(f"Standard deviation before scaling: {X_iris.std().values.round(2)}")
print(f"Standard deviation after scaling:  {X_scaled.std(axis=0).round(2)}")

#### Step 4: Instantiate a KMeans Object

Iris has three species, so we will try $k = 3$. Step 9 shows how to choose $k$ when you do not know how many groups to expect.

In [ ]:
# n_clusters is k, the number of clusters to find
# n_init=10 runs the algorithm 10 times from different starting centroids and keeps the best result
# random_state makes the result the same every time the notebook is run
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)

#### Step 5: Fit The Model

Notice that `fit` only receives `X_scaled`. Every other notebook in this series also passed a target (`y`). There is no `y` here, which is what makes this unsupervised learning.

In [ ]:
kmeans.fit(X_scaled)

Once fitted, the `kmeans` object stores what it learned as **attributes**. Scikit-learn ends the name of every learned attribute with an underscore (`_`).

In [ ]:
print(f"Centroids (in scaled units):\n{kmeans.cluster_centers_}")
print(f"Clusters of the first 5 flowers: {kmeans.labels_[:5]}")
print(f"Iterations until the centroids stopped moving: {kmeans.n_iter_}")

#### Step 6: Predict Values

In [ ]:
# The cluster each flower was assigned to during fitting
cluster_labels = kmeans.labels_

# Predict the cluster for a new data point
new_flower = pd.DataFrame({
    'PetalLengthCm': [4.5],
    'PetalWidthCm': [1.5]
})
# Use transform (not fit_transform) so the new flower is scaled the same way as the training data
new_flower_scaled = scaler.transform(new_flower)
new_flower['cluster'] = kmeans.predict(new_flower_scaled)
print(f"New flower is:\n{new_flower}")

> [!Important]
> The cluster numbers (0, 1 and 2) are only names. Cluster 0 does not mean the first species in the dataset. If you change `random_state`, the same groups can be given different numbers. Step 8 shows which species ended up in each cluster.

#### Step 7: Plot The Clusters

In [ ]:
# Convert the centroids from scaled units back to centimetres so they line up with the original data
centroids = scaler.inverse_transform(kmeans.cluster_centers_)

# Plot each flower coloured by its cluster, the centroids (X) and the new flower (diamond)
plt.scatter(X_iris.PetalLengthCm, X_iris.PetalWidthCm, c=cluster_labels, s=40, vmin=0, vmax=2)
plt.scatter(centroids[:, 0], centroids[:, 1], c='black', marker='X', s=200, label='Centroids')
plt.scatter(new_flower.PetalLengthCm, new_flower.PetalWidthCm, c=new_flower['cluster'], vmin=0, vmax=2, marker='D', s=100, edgecolors='black', label='New flower')
plt.xlabel("Petal length (cm)")
plt.ylabel("Petal width (cm)")
plt.colorbar(label="Cluster", ticks=[0, 1, 2])
plt.legend()
plt.show()

The model found the setosa group and split the larger flowers into two clusters. Each centroid sits at the mean position of the flowers in its cluster.

#### Step 8: Evaluate The Model

There is no target, so there is no accuracy to calculate. Instead, K-Means is evaluated on how tight and well separated its clusters are:

- **Inertia** is the sum of the squared distances from each data point to its own centroid. Lower inertia means tighter clusters. This is the same idea as the sum of squared errors used to calculate cost in notebooks 1 to 3.
- **Silhouette score** compares how close each point is to its own cluster with how close it is to the nearest other cluster. It ranges from -1 to 1. A score close to 1 means the clusters are well separated.

In [ ]:
# Get evaluative data from the model
print(f"Inertia: {kmeans.inertia_:.2f}")
print(f"Silhouette score: {silhouette_score(X_scaled, cluster_labels):.3f}")

In [ ]:
# Optional: Manually calculate the inertia
# Each flower's centroid is found using its cluster label
flower_centroids = kmeans.cluster_centers_[cluster_labels]
# Squared distance from each flower to its own centroid
squared_distances = ((X_scaled - flower_centroids) ** 2).sum(axis=1)

print(f"Manual inertia: {squared_distances.sum():.2f}")

#### Check The Clusters Against The Real Species

The model never saw the species. This dataset happens to include them, so we can check how well the clusters match. In a real unsupervised problem this check is usually not possible, which is why inertia and silhouette score matter.

In [ ]:
# Count how many flowers of each species ended up in each cluster
comparison = pd.crosstab(cluster_labels, species, rownames=['Cluster'], colnames=['Species'])
print(comparison)

# The most common species in each cluster counts as a match
matches = comparison.max(axis=1).sum()
print(f"\n{matches} of {len(species)} flowers ({matches / len(species):.0%}) are in a cluster dominated by their own species")

Setosa forms its own cluster with no mistakes. The flowers in the wrong cluster are versicolor and virginica flowers from the area where those two species overlap (see the plot in Step 2).

#### Step 9: Choose k With The Elbow Method

In a real project you usually do not know how many clusters to look for. The **elbow method** fits the model for a range of $k$ values and plots the inertia for each one.

In [ ]:
k_values = range(1, 11)
inertias = []

for k in k_values:
    model = KMeans(n_clusters=k, n_init=10, random_state=42)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.plot(k_values, inertias, marker='o')
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.xticks(k_values)
plt.show()

Inertia always goes down as $k$ goes up. If $k$ equalled the number of flowers, every flower would be its own centroid and the inertia would be 0, which tells you nothing. Look for the point where adding another cluster stops making a big difference. Here the inertia drops from 300 to 54 to 18 for $k$ = 1, 2 and 3, then only to 12 for $k = 4$. The bend (the "elbow") is at $k = 3$.

The silhouette score gives a second opinion.

In [ ]:
k_values = range(2, 11)  # The silhouette score needs at least 2 clusters
scores = []

for k in k_values:
    model = KMeans(n_clusters=k, n_init=10, random_state=42)
    model.fit(X_scaled)
    scores.append(silhouette_score(X_scaled, model.labels_))

plt.plot(k_values, scores, marker='o')
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.title("Silhouette Score For Each k")
plt.xticks(k_values)
plt.show()

The silhouette score is highest at $k = 2$, not $k = 3$. Setosa sits far from the other flowers while versicolor and virginica overlap, so from the data alone, two well separated groups is the strongest answer. Knowing there are three species is domain knowledge the model does not have. Choosing $k$ often needs both the metrics and an understanding of the problem.

#### Limitations Of K-Means

- You must choose $k$ before training.
- Every point is forced into a cluster, including outliers.
- It works best when clusters are roughly round and of similar size.
- The result depends on the starting centroids. Setting `n_init=10` runs the algorithm 10 times and keeps the result with the lowest inertia.
- Features on different scales must be scaled first (Step 3).

#### Step 10 (Optional): Watch The Algorithm Step By Step

The code below runs the assign and update steps from the pseudocode one at a time. Each plot shows where the centroids are at the start of the step and the clusters they create. The title shows how far the centroids then move. The code deliberately starts all three centroids on the largest virginica flowers (top right) so you can watch them spread out.

Scikit-learn uses a method called k-means++ to choose better starting centroids, which is why it needed fewer iterations (see `kmeans.n_iter_` in Step 5).

In [ ]:
# Start with three of the largest virginica flowers as the centroids (a poor start, so the centroids must move)
demo_centroids = X_scaled[[118, 117, 105]]

plt.figure(figsize=(16, 7))
for step in range(10):
    # Assign: each flower joins the cluster of its nearest centroid
    distances = np.array([[np.linalg.norm(flower - centroid) for centroid in demo_centroids] for flower in X_scaled])
    step_labels = distances.argmin(axis=1)

    # Update: move each centroid to the mean of the flowers in its cluster
    new_centroids = np.array([X_scaled[step_labels == cluster].mean(axis=0) for cluster in range(3)])
    distance_moved = np.linalg.norm(new_centroids - demo_centroids)

    # Plot the clusters and the centroids for this step (converted back to centimetres)
    centroids_cm = scaler.inverse_transform(demo_centroids)
    plt.subplot(2, 5, step + 1)
    plt.scatter(X_iris.PetalLengthCm, X_iris.PetalWidthCm, c=step_labels, s=15, vmin=0, vmax=2)
    plt.scatter(centroids_cm[:, 0], centroids_cm[:, 1], c='black', marker='X', s=150)
    plt.title(f"Step {step + 1} (next move {distance_moved:.2f})")

    demo_centroids = new_centroids

plt.tight_layout()
plt.show()

One centroid travels all the way down to the setosa group. The distance the centroids move gets smaller each step until it reaches 0 at step 10. At that point the algorithm has **converged**, and the clusters match the scikit-learn result in Step 7 (the cluster numbers may differ).

#### Extension Tasks

- Change `n_clusters` to 2, then 4. How do the clusters and the comparison table in Step 8 change?
- Use all four measurements: `X_iris = iris[['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']]`. Add the two sepal measurements to `new_flower`, and in Step 7 plot `centroids[:, 2]` and `centroids[:, 3]` (the petal columns). Skip Step 10, which only works with two features. Does more data always give clusters that better match the species?
- Apply K-Means to `./data/insects.csv` (read it with `sep='\t'`) using `latitude` and `wingsize` with $k = 2$. Compare the clusters with the `sex` column, then with the `continent` column. What did the model find without being told?